# Step 1 — Build BERTopic Model

Edit the **configuration cell** below, then run all cells.
The trained model, annotated data, and run parameters are saved together under `output/<run_name>/`
so you can keep multiple models and load any of them in `02_analyze_patterns.ipynb`.

In [ ]:
# ============================================================
# CONFIGURATION — edit this cell, then run all cells
# ============================================================

# Data source:
#   True  -> read from local CSV (mock data, development)
#   False -> connect via turbodbc (production database)
USE_LOCAL_DATA = True

# Embedding model:
#   "default"          -> all-MiniLM-L6-v2                      (fast, general-purpose)
#   "clinical_radiology" -> Bio_ClinicalBERT                    (domain-specific, slower)
#   "embeddinggemma"   -> embeddinggemma-300m-medical            (domain-specific, larger)
EMBEDDING = "default"

# Stop-word list:
#   "english"             -> standard sklearn English list
#   "medical_extended"    -> English + comprehensive medical/admin terms (recommended for production)
#   "radiology_stopwords" -> English + radiology/imaging terms
STOPWORDS = "english"

# UMAP — reduce n_neighbors for small datasets
#   Mock data (~20 docs):    n_neighbors=5
#   Production (1000+ docs): n_neighbors=15
UMAP_N_NEIGHBORS = 5
UMAP_N_COMPONENTS = 5

# HDBSCAN — reduce min_cluster_size for small datasets
#   Mock data:   min_cluster_size=3, min_samples=2
#   Production:  min_cluster_size=30, min_samples=5
HDBSCAN_MIN_CLUSTER_SIZE = 3
HDBSCAN_MIN_SAMPLES = 2

# Optional custom name for this run.
# Leave empty to auto-generate from parameters, e.g. "default__english__20260604_143022".
RUN_TAG = ""

In [ ]:
import pandas as pd
import numpy as np
import json
import os
import datetime
from nltk.tokenize import WordPunctTokenizer
from bertopic import BERTopic
from bertopic.representation import KeyBERTInspired
from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import CountVectorizer
from umap import UMAP
from hdbscan import HDBSCAN

In [ ]:
if USE_LOCAL_DATA:
    print("Reading mock data from local CSV...")
    df = pd.read_csv("mock_clinical_data.csv")
else:
    import configparser
    from turbodbc import connect, make_options
    config = configparser.ConfigParser()
    config.read('./config.ini')
    dl_config = config['DataLoch']
    dl_conn = connect(
        driver=dl_config['driver'],
        server=dl_config['server'],
        port=dl_config['port'],
        database=dl_config['database'],
        uid=dl_config['uid'],
        pwd=dl_config['pwd'])
    with open("query.sql") as f:
        query = f.read()
    df = pd.read_sql_query(query, con=dl_conn)
    seed = 1337
    df = df[df.apply(lambda x: x["reptext"] != '<c>', axis=1)].sample(100000, random_state=seed)

print(f"Loaded {len(df)} records.")
display(df.head())

In [ ]:
punkt = WordPunctTokenizer()

# Lowercase
df["reptext"] = df["reptext"].str.lower()
# Drop blanks, placeholder tokens, and single-token entries
mask = (
    ~df["reptext"].isin(["", "<c>", " "])
    & df["reptext"].apply(lambda t: len(punkt.tokenize(t)) > 1)
)
df = df[mask].reset_index(drop=True)
docs = df["reptext"].tolist()
print(f"After filtering: {len(docs)} documents.")

In [ ]:
# Embedding model
if EMBEDDING == "clinical_radiology":
    embedding_model = SentenceTransformer("emilyalsentzer/Bio_ClinicalBERT")
    print("Embedding: Bio_ClinicalBERT")
elif EMBEDDING == "embeddinggemma":
    embedding_model = SentenceTransformer("sentence-transformers/embeddinggemma-300m-medical")
    print("Embedding: embeddinggemma-300m-medical")
else:
    embedding_model = SentenceTransformer("all-MiniLM-L6-v2")
    print("Embedding: all-MiniLM-L6-v2")

# Stop-words
_base = list(CountVectorizer(stop_words="english").get_stop_words())
stopword_sets = {
    "english": _base,
    "medical_extended": list(set(_base) | {
        "patient", "patients", "report", "showed", "shows", "history", "clinical",
        "indicated", "clinic", "hospital", "admission", "discharge",
        "present", "past", "status", "reported", "denies", "chief",
        "complaint", "noted", "examination", "assessment", "plan", "treatment",
        "daily", "bid", "tid", "qid", "mg", "ml", "tablet", "capsule", "pills", "med",
        "medication", "medications", "prescribed", "ordered", "intake",
        "doctor", "nurse", "physician", "provider", "referred", "consult",
        "follow", "up", "date", "time", "year", "old", "male", "female", "age",
    }),
    "radiology_stopwords": list(set(_base) | {
        "imaging", "scan", "report", "findings", "radiology", "ct", "mri",
        "x-ray", "contrast", "study", "exam", "views", "series",
    }),
}
selected_stopwords = stopword_sets.get(STOPWORDS, stopword_sets["english"])
print(f"Stop-words: {STOPWORDS} ({len(selected_stopwords)} words)")
vectorizer_model = CountVectorizer(ngram_range=(1, 3), stop_words=selected_stopwords)

# Sub-models
umap_model = UMAP(
    n_neighbors=UMAP_N_NEIGHBORS, n_components=UMAP_N_COMPONENTS,
    min_dist=0.0, metric="cosine", random_state=42,
)
hdbscan_model = HDBSCAN(
    min_cluster_size=HDBSCAN_MIN_CLUSTER_SIZE, min_samples=HDBSCAN_MIN_SAMPLES,
    metric="euclidean", prediction_data=True,
)

topic_model = BERTopic(
    embedding_model=embedding_model,
    umap_model=umap_model,
    hdbscan_model=hdbscan_model,
    vectorizer_model=vectorizer_model,
    representation_model=KeyBERTInspired(),
    nr_topics="auto",
    verbose=True,
)
print("BERTopic configured.")

In [ ]:
# Pre-compute embeddings once.
# To iterate on topics without re-embedding, re-run only the cells below this one.
print(f"Encoding {len(docs)} documents with '{EMBEDDING}'...")
embeddings = embedding_model.encode(docs, show_progress_bar=True)
print(f"Embeddings shape: {embeddings.shape}")

In [ ]:
print(f"Fitting BERTopic on {len(docs)} documents...")
topics, probs = topic_model.fit_transform(docs, embeddings)
df["topic"] = topics
print("Done.\n")

topic_info = topic_model.get_topic_info()
display(topic_info)

print("\nTop words per topic:")
for tid in sorted(set(topics)):
    words = topic_model.get_topic(tid)
    label = f"Topic {tid}" if tid != -1 else "Outliers (-1)"
    print(f"  {label}: {[w for w, _ in (words or [])[:8]]}")

## Topic visualisations

Inspect the topics below before deciding whether to save. If the model looks poor, adjust the configuration cell and re-run without proceeding to the save cell.

In [ ]:
# Top-words bar chart — one bar per topic, ranked by c-TF-IDF score
n_real_topics = len([t for t in set(topics) if t != -1])
top_n = min(n_real_topics, 15)  # show at most 15 topics at once
topic_model.visualize_barchart(top_n_topics=top_n, n_words=10, title="Top words per topic")

In [ ]:
# 2-D topic scatter — shows how well-separated the topics are
# Needs at least 3 topics; skipped automatically for tiny mock runs
if n_real_topics >= 3:
    topic_model.visualize_topics(title="Topic map (2-D)")
else:
    print(f"Only {n_real_topics} topic(s) found — scatter plot needs at least 3.")

In [ ]:
# Auto-generate run name from parameters + timestamp if no custom tag given
_ts = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
run_name = RUN_TAG.strip() or f"{EMBEDDING}__{STOPWORDS}__{_ts}"
run_dir = os.path.join("output", run_name)
os.makedirs(run_dir, exist_ok=True)

# Save BERTopic model
model_path = os.path.join(run_dir, "model")
topic_model.save(model_path, serialization="safetensors", save_ctfidf=True)

# Save annotated data as Parquet
data_path = os.path.join(run_dir, "annotated_data.parquet")
df.to_parquet(data_path, index=False)

# Save run config + metadata so 02_analyze_patterns.ipynb can reload correctly
n_topics = len([t for t in set(topics) if t != -1])
run_config = {
    "run_name": run_name,
    "embedding": EMBEDDING,
    "stopwords": STOPWORDS,
    "use_local_data": USE_LOCAL_DATA,
    "umap_n_neighbors": UMAP_N_NEIGHBORS,
    "umap_n_components": UMAP_N_COMPONENTS,
    "hdbscan_min_cluster_size": HDBSCAN_MIN_CLUSTER_SIZE,
    "hdbscan_min_samples": HDBSCAN_MIN_SAMPLES,
    "trained_on": datetime.datetime.now().isoformat(timespec="seconds"),
    "n_docs": len(docs),
    "n_topics": n_topics,
}
with open(os.path.join(run_dir, "run_config.json"), "w") as f:
    json.dump(run_config, f, indent=2)

print(f"Saved to: {run_dir}/")
print(f"  model/               <- BERTopic model")
print(f"  annotated_data.parquet")
print(f"  run_config.json      <- {n_topics} topics, {len(docs)} docs")
print(f"\nTo analyse: set MODEL_TO_LOAD = \"{run_name}\" in 02_analyze_patterns.ipynb")